# Cow Behavior Classification with Vision Transformer

Robust training/evaluation mirror of the notebook workflow.

- Trains a ViT classifier on behavior crops
- Uses a fixed held-out test split
- Runs 5-fold cross-validation on the remaining training data
- Retrains a final model on all non-test data
- Saves organized visual artifacts to `artifacts/figures/vit_classifier/`

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedKFold
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [2]:
# Configuration
DATA_ROOT = Path("workdir/crops_raw")
MODEL_CHECKPOINT = "google/vit-base-patch16-224-in21k"
MODEL_OUTPUT_DIR = Path("artifacts/models/cow-behavior-vit")
FIGURES_DIR = Path("artifacts/figures/vit_classifier")
CV_FIGURES_DIR = FIGURES_DIR / "cv"
TRAINING_RUNS_DIR = Path("artifacts/runs/cow-behavior-vit")
CV_RUNS_DIR = TRAINING_RUNS_DIR / "cv"
FINAL_RUN_DIR = TRAINING_RUNS_DIR / "final"

EPOCHS = 10
LEARNING_RATE = 5e-5
BATCH_TRAIN = 32
BATCH_EVAL = 64
SEED = 42
SAVE_MODEL = True
TEST_SIZE = 0.15
NUM_FOLDS = 5
EARLY_STOPPING_PATIENCE = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_ROOT}")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CV_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_RUNS_DIR.mkdir(parents=True, exist_ok=True)
CV_RUNS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

Using CUDA: NVIDIA GeForce RTX 4080


In [3]:
# Dataset loading and splitting
def load_raw_behavior_dataset(data_root: Path) -> Dataset:
    return load_dataset("imagefolder", data_dir=str(data_root))["train"]


def make_holdout_split(raw_dataset: Dataset) -> DatasetDict:
    split = raw_dataset.train_test_split(
        test_size=TEST_SIZE,
        stratify_by_column="label",
        seed=SEED,
    )
    return split


raw_dataset = load_raw_behavior_dataset(DATA_ROOT)
dataset = make_holdout_split(raw_dataset)

print(
    f"Dataset splits -> train_for_cv: {len(dataset['train']):,}, "
    f"held_out_test: {len(dataset['test']):,}"
)

Resolving data files:   0%|          | 0/25322 [00:00<?, ?it/s]

Dataset splits -> train_for_cv: 21,523, held_out_test: 3,799


In [4]:
# Class distribution summary
class_names = raw_dataset.features["label"].names
train_counts = Counter(dataset["train"]["label"])
test_counts = Counter(dataset["test"]["label"])
class_distribution = pd.DataFrame(
    {
        "class": class_names,
        "train_count": [train_counts[i] for i in range(len(class_names))],
        "test_count": [test_counts[i] for i in range(len(class_names))],
        "train_percent": [
            100.0 * train_counts[i] / len(dataset["train"])
            for i in range(len(class_names))
        ],
        "test_percent": [
            100.0 * test_counts[i] / len(dataset["test"])
            for i in range(len(class_names))
        ],
    }
).round(2)
print("Class distribution (train/test holdout split):")
print(class_distribution.to_string(index=False))

Class distribution (train/test holdout split):
         class  train_count  test_count  train_percent  test_percent
drinking water          632         112           2.94          2.95
      foraging         4853         856          22.55         22.53
    lying down         3840         678          17.84         17.85
    rumination         5167         912          24.01         24.01
         stand         7031        1241          32.67         32.67


In [5]:
# Preprocessing and metrics helpers
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in id2label.items()}
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()


def preprocess(examples: dict) -> dict:
    images = [img.convert("RGB") for img in examples["image"]]
    inputs = processor(images)
    return {
        "pixel_values": inputs["pixel_values"],
        "labels": examples["label"],
    }


def prepare_dataset(split_dataset: Dataset) -> Dataset:
    prepared = split_dataset.map(
        preprocess, batched=True, remove_columns=["image", "label"]
    )
    prepared.set_format("torch", columns=["pixel_values", "labels"])
    return prepared


def compute_metrics(eval_pred) -> dict[str, float]:
    logits = (
        eval_pred.predictions[0]
        if isinstance(eval_pred.predictions, tuple)
        else eval_pred.predictions
    )
    preds = np.asarray(logits).argmax(-1)
    references = np.asarray(eval_pred.label_ids)
    acc = float((preds == references).mean())
    f1_weighted = float(
        precision_recall_fscore_support(
            references,
            preds,
            average="weighted",
            zero_division=0,
        )[2]
    )
    return {"accuracy": acc, "f1_weighted": f1_weighted}


def make_model() -> AutoModelForImageClassification:
    return AutoModelForImageClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(id2label),
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    ).to(device)


def make_training_args(output_dir: Path, use_eval: bool = True) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        learning_rate=LEARNING_RATE,
        weight_decay=0.05,
        warmup_ratio=0.05,
        eval_strategy="epoch" if use_eval else "no",
        save_strategy="epoch" if use_eval else "no",
        load_best_model_at_end=use_eval,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,
        bf16=use_bf16,
        fp16=(not use_bf16 and torch.cuda.is_available()),
        report_to="none",
        save_total_limit=1,
        seed=SEED,
    )


def make_trainer(
    train_dataset: Dataset,
    eval_dataset: Dataset | None,
    output_dir: Path,
    use_early_stopping: bool = True,
) -> Trainer:
    return Trainer(
        model=make_model(),
        args=make_training_args(output_dir, use_eval=eval_dataset is not None),
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        processing_class=processor,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE,
            )
        ]
        if use_early_stopping and eval_dataset is not None
        else [],
    )


def evaluate_predictions(
    trainer: Trainer, eval_dataset: Dataset
) -> tuple[np.ndarray, np.ndarray, dict[str, float]]:
    predictions = trainer.predict(eval_dataset)
    logits = (
        predictions.predictions[0]
        if isinstance(predictions.predictions, tuple)
        else predictions.predictions
    )
    y_true = np.asarray(predictions.label_ids)
    y_pred = np.asarray(logits).argmax(-1)
    metrics = {
        "accuracy": float((y_pred == y_true).mean()),
        "f1_weighted": float(
            precision_recall_fscore_support(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )[2]
        ),
    }
    return y_true, y_pred, metrics


def save_json(data: dict | list, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [6]:
# Cross-validation
def run_cross_validation(
    train_dataset_raw: Dataset,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    labels = np.array(train_dataset_raw["label"])
    splitter = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    fold_rows: list[dict[str, float | int]] = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        splitter.split(np.zeros(len(labels)), labels),
        start=1,
    ):
        print(f"\n===== Fold {fold_idx}/{NUM_FOLDS} =====")
        fold_train_raw = train_dataset_raw.select(train_idx.tolist())
        fold_val_raw = train_dataset_raw.select(val_idx.tolist())
        fold_train = prepare_dataset(fold_train_raw)
        fold_val = prepare_dataset(fold_val_raw)

        trainer = make_trainer(
            train_dataset=fold_train,
            eval_dataset=fold_val,
            output_dir=CV_RUNS_DIR / f"fold_{fold_idx}",
        )
        train_result = trainer.train()
        _, _, fold_metrics = evaluate_predictions(trainer, fold_val)

        fold_row = {
            "fold": fold_idx,
            "train_samples": len(fold_train_raw),
            "validation_samples": len(fold_val_raw),
            "accuracy": fold_metrics["accuracy"],
            "f1_weighted": fold_metrics["f1_weighted"],
            "train_runtime_sec": float(train_result.metrics.get("train_runtime", 0.0)),
            "train_steps_per_sec": float(
                train_result.metrics.get("train_steps_per_second", 0.0)
            ),
            "train_samples_per_sec": float(
                train_result.metrics.get("train_samples_per_second", 0.0)
            ),
        }
        fold_rows.append(fold_row)
        print(
            "Fold metrics -> "
            f"accuracy: {fold_row['accuracy']:.4f}, "
            f"weighted_f1: {fold_row['f1_weighted']:.4f}, "
            f"runtime: {fold_row['train_runtime_sec'] / 60.0:.2f} min"
        )

    fold_results = pd.DataFrame(fold_rows)
    summary = pd.DataFrame(
        {
            "metric": ["accuracy", "f1_weighted", "train_runtime_sec"],
            "mean": [
                fold_results["accuracy"].mean(),
                fold_results["f1_weighted"].mean(),
                fold_results["train_runtime_sec"].mean(),
            ],
            "std": [
                fold_results["accuracy"].std(ddof=1),
                fold_results["f1_weighted"].std(ddof=1),
                fold_results["train_runtime_sec"].std(ddof=1),
            ],
        }
    )
    return fold_results, summary


cv_fold_results, cv_summary = run_cross_validation(dataset["train"])
cv_fold_results.to_csv(CV_RUNS_DIR / "fold_metrics.csv", index=False)
cv_summary.to_csv(CV_RUNS_DIR / "summary_metrics.csv", index=False)
save_json(cv_fold_results.to_dict(orient="records"), CV_RUNS_DIR / "fold_metrics.json")
save_json(cv_summary.to_dict(orient="records"), CV_RUNS_DIR / "summary_metrics.json")

print("\nCross-validation summary:")
print(cv_summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


===== Fold 1/5 =====


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.816719,0.456482,0.834843,0.832253
2,0.356159,0.329608,0.892218,0.892019
3,0.191804,0.304154,0.905923,0.905134
4,0.108481,0.348859,0.903368,0.902206
5,0.055191,0.379194,0.907782,0.908173
6,0.027509,0.420905,0.912892,0.912007
7,0.013848,0.395148,0.923113,0.923082
8,0.007130,0.435397,0.917073,0.916997
9,0.006290,0.440619,0.919164,0.918905


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold metrics -> accuracy: 0.9054, weighted_f1: 0.9053, runtime: 11.99 min

Cross-validation summary:
           metric      mean      std
         accuracy    0.9201   0.0090
      f1_weighted    0.9200   0.0090
train_runtime_sec 1276.5077 312.8693


In [7]:
# CV figure
def save_cv_metrics_figure(fold_results: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(fold_results))
    width = 0.35

    ax.bar(x - width / 2, fold_results["accuracy"], width, label="Accuracy", alpha=0.9)
    ax.bar(
        x + width / 2,
        fold_results["f1_weighted"],
        width,
        label="Weighted F1",
        alpha=0.9,
    )

    ax.set_title("5-Fold Cross-Validation Metrics")
    ax.set_xlabel("Fold")
    ax.set_ylabel("Score")
    ax.set_xticks(x)
    ax.set_xticklabels([str(fold) for fold in fold_results["fold"]])
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.legend()

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


save_cv_metrics_figure(cv_fold_results, CV_FIGURES_DIR / "cv_fold_metrics.png")

In [8]:
# Final train on all non-test data
final_train_dataset = prepare_dataset(dataset["train"])
final_test_dataset = prepare_dataset(dataset["test"])

trainer = make_trainer(
    train_dataset=final_train_dataset,
    eval_dataset=None,
    output_dir=FINAL_RUN_DIR,
    use_early_stopping=False,
)

trainer.train()

Map:   0%|          | 0/21523 [00:00<?, ? examples/s]

The channel dimension is ambiguous. Got image shape torch.Size([3, 11, 3]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
500,0.847295
1000,0.403287
1500,0.256104
2000,0.171657
2500,0.103837
3000,0.062592
3500,0.039589
4000,0.021560
4500,0.011313
5000,0.006800


TrainOutput(global_step=6730, training_loss=0.14352652023575457, metrics={'train_runtime': 1414.1996, 'train_samples_per_second': 152.192, 'train_steps_per_second': 4.759, 'total_flos': 1.6679049379822449e+19, 'train_loss': 0.14352652023575457, 'epoch': 10.0})

In [9]:
# Final held-out test evaluation
y_true, y_pred, test_metrics = evaluate_predictions(trainer, final_test_dataset)

print("Held-out test metrics:")
print(test_metrics)
print("\nClassification report:")
classification_report_text = classification_report(
    y_true,
    y_pred,
    target_names=list(id2label.values()),
    zero_division=0,
)
print(classification_report_text)

save_json(test_metrics, FINAL_RUN_DIR / "held_out_test_metrics.json")
(FINAL_RUN_DIR / "held_out_test_classification_report.txt").write_text(
    classification_report_text,
    encoding="utf-8",
)

if SAVE_MODEL:
    trainer.save_model(str(MODEL_OUTPUT_DIR))
    processor.save_pretrained(str(MODEL_OUTPUT_DIR))
    print(f"Saved final model to {MODEL_OUTPUT_DIR}")

Held-out test metrics:
{'accuracy': 0.928138983943143, 'f1_weighted': 0.9278836705875583}

Classification report:
                precision    recall  f1-score   support

drinking water       0.95      0.97      0.96       112
      foraging       0.95      0.97      0.96       856
    lying down       0.89      0.86      0.87       678
    rumination       0.89      0.89      0.89       912
         stand       0.96      0.96      0.96      1241

      accuracy                           0.93      3799
     macro avg       0.93      0.93      0.93      3799
  weighted avg       0.93      0.93      0.93      3799



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model to artifacts/models/cow-behavior-vit


In [10]:
# Final test figures
def save_confusion_matrix_figure(
    y_true_values: np.ndarray,
    y_pred_values: np.ndarray,
    labels: list[str],
    output_path: Path,
) -> None:
    cm = confusion_matrix(y_true_values, y_pred_values)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=labels,
        yticklabels=labels,
        title="Confusion Matrix (Normalized)",
        ylabel="True label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = cm_norm[i, j]
            text_color = "white" if value > 0.5 else "black"
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", color=text_color)

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_precision_recall_figure(
    y_true_values: np.ndarray,
    y_pred_values: np.ndarray,
    labels: list[str],
    output_path: Path,
) -> None:
    precision, recall, _, _ = precision_recall_fscore_support(
        y_true_values,
        y_pred_values,
        average=None,
        zero_division=0,
    )

    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(11, 7))
    bars_precision = ax.bar(
        x - width / 2,
        precision,
        width,
        label="Precision",
        alpha=0.85,
    )
    bars_recall = ax.bar(
        x + width / 2,
        recall,
        width,
        label="Recall",
        alpha=0.85,
    )

    ax.set_xlabel("Behavior")
    ax.set_ylabel("Score")
    ax.set_title("Precision and Recall by Behavior")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylim(0, 1.0)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)

    for bar_group in [bars_precision, bars_recall]:
        for bar in bar_group:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height,
                f"{height:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def save_sample_predictions_grid(
    split_data: Dataset,
    labels_map: dict[int, str],
    trained_model: AutoModelForImageClassification,
    output_path: Path,
    samples_per_class: int = 2,
) -> None:
    sample_indices: list[int] = []
    by_class: dict[int, list[int]] = {i: [] for i in labels_map.keys()}

    for idx, label in enumerate(split_data["labels"]):
        label_value = int(label.item() if hasattr(label, "item") else label)
        if len(by_class[label_value]) < samples_per_class:
            by_class[label_value].append(idx)

    for class_id in sorted(by_class.keys()):
        sample_indices.extend(by_class[class_id])

    n = min(len(sample_indices), 10)
    if n == 0:
        return

    fig, axes = plt.subplots(2, 5, figsize=(16, 7))
    axes = axes.flatten()

    for i in range(10):
        axes[i].axis("off")

    trained_model.eval()

    for i, sample_idx in enumerate(sample_indices[:n]):
        pixel_values = split_data[sample_idx]["pixel_values"].unsqueeze(0).to(device)
        true_label = int(split_data[sample_idx]["labels"].item())

        with torch.no_grad():
            logits = trained_model(pixel_values).logits
            pred_label = int(logits.argmax(-1).item())
            confidence = float(F.softmax(logits, dim=-1).max().item())

        image = pixel_values.squeeze().cpu().numpy().transpose(1, 2, 0)
        image = np.clip(image * 0.5 + 0.5, 0, 1)

        axes[i].imshow(image)
        axes[i].axis("off")
        color = "green" if true_label == pred_label else "red"
        axes[i].set_title(
            f"T: {labels_map[true_label]}\nP: {labels_map[pred_label]} ({confidence:.2f})",
            fontsize=9,
            color=color,
        )

    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


save_confusion_matrix_figure(
    y_true_values=y_true,
    y_pred_values=y_pred,
    labels=list(id2label.values()),
    output_path=FIGURES_DIR / "confusion_matrix.png",
)

save_precision_recall_figure(
    y_true_values=y_true,
    y_pred_values=y_pred,
    labels=list(id2label.values()),
    output_path=FIGURES_DIR / "precision_recall_by_behavior.png",
)

save_sample_predictions_grid(
    split_data=final_test_dataset,
    labels_map=id2label,
    trained_model=trainer.model,
    output_path=FIGURES_DIR / "sample_predictions_grid.png",
)

print(f"Saved figures to {FIGURES_DIR}")

Saved figures to artifacts/figures/vit_classifier
